## 03 — Split agrupado dos fundos (v3.2)
O split é feito por fundo e hash. Com 250 fundos A do SDSS, a proporção 68/16/16
preserva 40 fundos independentes tanto na validação quanto no teste. Categoria B é
exclusiva do treino e categoria C é excluída.

In [1]:
from google.colab import drive
from pathlib import Path
import hashlib, json, shutil, time
import numpy as np
import pandas as pd

def log_etapa(mensagem):
    print(f"[{time.strftime('%H:%M:%S')}] {mensagem}", flush=True)

drive.mount('/content/drive', force_remount=True)
BACKUP_DIR = Path('/content/drive/MyDrive/tcc-satellite-streaks')
PROC = Path('data/processed_v3'); PROC.mkdir(parents=True, exist_ok=True)
origem = BACKUP_DIR/'data/processed_v3/manifest_curado_v3.csv'
shutil.copy2(origem, PROC/'manifest_curado_v3.csv')
df = pd.read_csv(PROC/'manifest_curado_v3.csv')

SEED_SPLIT = 42
PROPORCOES = {'train':0.68, 'val':0.16, 'test':0.16}
MIN_A_VAL_TEST = 40

Mounted at /content/drive


## 1. Divisão determinística por fonte e hash

In [2]:
assert set(df.categoria) <= {'A','B','C'}
assert df.sha256_processado.is_unique
df_a = df[df.categoria=='A'].copy()
df_b = df[df.categoria=='B'].copy()
assert len(df_a) > 0

def alocar_maior_resto(total, proporcoes):
    bruto = {k:total*v for k,v in proporcoes.items()}
    base = {k:int(np.floor(v)) for k,v in bruto.items()}
    faltam = total-sum(base.values())
    ordem = sorted(proporcoes, key=lambda k:(bruto[k]-base[k],k), reverse=True)
    for k in ordem[:faltam]: base[k] += 1
    return base

partes=[]
for fonte, grupo in df_a.groupby('fonte', sort=True):
    grupo=grupo.sort_values('sha256_processado').copy()
    seed=SEED_SPLIT+int(hashlib.sha256(fonte.encode()).hexdigest()[:8],16)
    rng=np.random.default_rng(seed)
    grupo=grupo.iloc[rng.permutation(len(grupo))].reset_index(drop=True)
    cont=alocar_maior_resto(len(grupo),PROPORCOES)
    grupo['split']=(['train']*cont['train']+['val']*cont['val']+['test']*cont['test'])
    partes.append(grupo)
df_a_split=pd.concat(partes,ignore_index=True)
df_b['split']='train'
df_split=pd.concat([df_a_split,df_b],ignore_index=True)

## 2. Gate local, publicação e verificação

In [4]:
assert df_split.id_fundo.is_unique
assert df_split.groupby('sha256_processado').split.nunique().max()==1
assert not ((df_split.categoria=='B')&(df_split.split!='train')).any()
assert not df_split.categoria.eq('C').any()

n_val=int((df_a_split.split=='val').sum())
n_test=int((df_a_split.split=='test').sum())
assert n_val>=MIN_A_VAL_TEST and n_test>=MIN_A_VAL_TEST, (
    f'Diversidade insuficiente: A-val={n_val}, A-test={n_test}'
)
print(pd.crosstab([df_split.split,df_split.categoria],df_split.fonte))

saida=PROC/'manifest_split_fundos_v3.csv'
df_split.sort_values(['split','categoria','fonte','id_fundo']).to_csv(saida,index=False)
hash_manifest=hashlib.sha256(saida.read_bytes()).hexdigest()
contrato={
    'versao':'v3.2','seed_split':SEED_SPLIT,'proporcoes':PROPORCOES,
    'sha256_manifest_split':hash_manifest,
    'contagens':df_split.split.value_counts().to_dict(),
    'contagens_A':df_a_split.split.value_counts().to_dict(),
}
contrato_path=PROC/'contrato_split_v3.json'
contrato_path.write_text(json.dumps(contrato,indent=2),encoding='utf-8')
resumo={
    'status':'SPLIT_LOCAL_APROVADO','hash_manifest':hash_manifest,
    'A_val':n_val,'A_test':n_test,
    'fundos_por_split':df_split.split.value_counts().to_dict(),
}
print(json.dumps(resumo,indent=2,ensure_ascii=False))

PUBLICAR_SPLIT_NO_DRIVE=True
if not PUBLICAR_SPLIT_NO_DRIVE:
    print('Publicação bloqueada. Confira o resumo e altere a chave para True.')
else:
    destino=BACKUP_DIR/'data/processed_v3';destino.mkdir(parents=True,exist_ok=True)
    log_etapa('Publicando manifest e contrato do split')
    shutil.copy2(saida,destino/saida.name)
    shutil.copy2(contrato_path,destino/contrato_path.name)
    assert hashlib.sha256((destino/saida.name).read_bytes()).hexdigest()==hash_manifest
    assert hashlib.sha256((destino/contrato_path.name).read_bytes()).hexdigest()==hashlib.sha256(contrato_path.read_bytes()).hexdigest()
    print(json.dumps({'status':'BACKUP_SPLIT_APROVADO','hash_manifest':hash_manifest},indent=2))

fonte            hubble  jwst  sdss
split categoria                    
test  A               0     0    40
train A               0     0   170
      B              67     8     0
val   A               0     0    40
{
  "status": "SPLIT_LOCAL_APROVADO",
  "hash_manifest": "9a2c2cd60ff69cd93d65824ae035ecbe4135f9cdf605b9bb527b6122f4daa67d",
  "A_val": 40,
  "A_test": 40,
  "fundos_por_split": {
    "train": 245,
    "val": 40,
    "test": 40
  }
}
[23:10:18] Publicando manifest e contrato do split
{
  "status": "BACKUP_SPLIT_APROVADO",
  "hash_manifest": "9a2c2cd60ff69cd93d65824ae035ecbe4135f9cdf605b9bb527b6122f4daa67d"
}
